# ComfyUI + Wan 2.2 5B Setup\n\nRun these cells in Google Colab with a GPU runtime. This starts ComfyUI, downloads the Wan model files, and exposes a temporary ComfyUI URL.

In [ ]:
!nvidia-smi

## Install ComfyUI

In [ ]:
%%bash\nset -e\ncd /content\nif [ ! -d ComfyUI ]; then\n  git clone https://github.com/comfyanonymous/ComfyUI.git\nfi\ncd /content/ComfyUI\npython -m pip install -r requirements.txt

## Download Wan 2.2 5B Model Files\n\nThese files are large. This can take a while.

In [ ]:
%%bash\nset -e\ncd /content/ComfyUI\nmkdir -p models/diffusion_models models/text_encoders models/vae\nwget -c -P models/diffusion_models https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors\nwget -c -P models/text_encoders https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors\nwget -c -P models/vae https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors\nfind models/diffusion_models models/text_encoders models/vae -maxdepth 1 -type f -printf '%p %k KB\\n'

## Install Cloudflared Tunnel\n\nThis gives a temporary public URL so you can open ComfyUI from your browser.

In [ ]:
%cd /content\n!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared\n!chmod +x cloudflared

## Start Tunnel\n\nRun this cell. Copy the `https://...trycloudflare.com` URL that appears in the output.

In [ ]:
import subprocess, time\n\ntunnel = subprocess.Popen(\n    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    text=True,\n)\n\nfor _ in range(60):\n    line = tunnel.stdout.readline()\n    print(line, end='')\n    if 'trycloudflare.com' in line:\n        break\n    time.sleep(1)

## Start ComfyUI\n\nRun this cell and leave it running. Open the tunnel URL from the previous cell.

In [ ]:
%cd /content/ComfyUI\n!python main.py --listen 0.0.0.0 --port 8188

## In ComfyUI\n\n1. Load `workflows/video/wan-text-to-video-ui.json` from this repo.\n2. Generate one short MP4 manually.\n3. Use `Save (API Format)`.\n4. Save the exported file locally as `workflows/video/wan-text-to-video-api.json`.\n5. Back in the repo, run `scripts/inspect_workflow.py` on that API JSON.